# 20 - Monitoring and recovery
Run the setup cell first, then the work cells in order. Use the WSL mlop-pj kernel.
These cells change cloud resources when you run them. A pending result needs a wait and a phase retry, not a new job.
Use notebooks 18-22 for the five main parts. Training and log waits can overlap with GitHub setup. See docs/RUN_FINISH.md.


In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert Path(sys.prefix).name == "mlop-pj", "Select the WSL mlop-pj kernel."
assert "microsoft" in os.uname().release.lower(), "Open this project in WSL."
status = json.loads((ROOT / "reports/azure-blob-upload.json").read_text())
assert status.get("data_uploaded") and status.get("all_hashes_match") and not status["errors"], "Finish 17 first."

def step(phase):
    print("Running:", phase, flush=True)
    process = subprocess.run([sys.executable, "-m", "cloudlayer.finish", phase],
                             cwd=ROOT, capture_output=True, text=True)
    if process.returncode:
        raise RuntimeError("Phase process failed. Check its stderr before retrying: " + process.stderr)
    result = json.loads(process.stdout)
    print(json.dumps(result, indent=2))
    if result["errors"]:
        raise RuntimeError(f"Check reports/finish-{phase}.json and the matching log in tmp. Stop this group.")
    if result.get("pending"):
        print("Still waiting. Read next in the report; do not recreate the resource or job.")
    return result

print("Root:", ROOT)
print("17 passed. Run this notebook work cells in order.")

Root: /mnt/c/Users/SN23341J001185/Documents/MyWork/ICT/ICTYear4_1/MLop/public_teaching_mlaiops/course/MLOP-PJ
17 passed. Run this notebook work cells in order.


## Controlled input drift
Run this local check before monitoring. It calibrates a maximum KS score on training windows, checks a held-out window, and injects a clear rating shift. Output: reports/input-drift.json. The report must say injected_detected: true. This is a controlled demonstration, not live or historical drift.


In [2]:
subprocess.run([sys.executable, "-m", "monitoring.drift"], cwd=ROOT, check=True)


{
  "method": "Maximum feature-wise two-sample KS statistic",
  "reference_rows": 1908,
  "recent_rows": 200,
  "calibration_rounds": 100,
  "threshold": 0.12566457023060795,
  "normal_score": 0.09936058700209643,
  "normal_flagged": false,
  "injected_score": 0.7878930817610064,
  "injected_detected": true,
  "injection": "Add 1.5 to the first four feature ratings, clipped to [0,5]",
  "seed": 42,
  "limits": [
    "Controlled input drift demonstration, not observed temporal drift.",
    "Threshold is calibrated on training windows; the normal holdout may still be flagged.",
    "The dataset has no timestamps or outcome labels. This does not prove concept drift.",
    "This offline check is not a live rolling-window monitor."
  ],
  "verified": true,
  "processed_sha256": "dd202f838ece1c4c2be3859d7e774afbf9cdfcfa6b999c8caa2adfd8725c79b1"
}


CompletedProcess(args=['/home/dew_perawis/.venvs/mlop-pj/bin/python', '-m', 'monitoring.drift'], returncode=0)

## Group 3 — Dashboard, real alert, and recovery
The dashboard reads Container App logs for requests, errors, server latency, model version, and uncertainty. Logs may arrive a few minutes after deployment. Run the monitor cell again if `pending: true`; continue only when `monitor_ready: true`.

Optional: create `tmp/alert-email.txt` with your own email before running monitor if you want email delivery. Otherwise, check the alert in Azure Monitor. Do not commit that file.

The failure cell changes the segment mapping on a separate staging app. Readiness still passes, while the semantic canary fails. The script restores the correct mapping in a `finally` block and checks recovery. It leaves the main API working. This creates an extra app which can scale to zero.

The alert cell checks actual Azure alert records. Creating an alert rule alone is not proof it fired. Save the report and an Azure screenshot when `alert_fired: true`.

In [3]:
monitor = step("monitor")

Running: monitor
{
  "checked_at": "2026-10-11T16:53:48.913917+00:00",
  "phase": "monitor",
  "errors": [],
  "monitor_ready": true,
  "dashboard_created": true,
  "dashboard_name": "MLOP Travel Style",
  "alert_created": true,
  "email_notifications_configured": false,
  "next": "Run the failure cell, then refresh evidence to confirm an actual fired alert."
}


In [4]:
monitor_status = json.loads((ROOT / "reports/finish-monitor.json").read_text())
assert monitor_status.get("monitor_ready") and not monitor_status["errors"], "Rerun the monitor cell after logs arrive."
failure = step("failure")
alert = step("alert")

Running: failure
{
  "checked_at": "2026-10-11T16:54:15.924979+00:00",
  "phase": "failure",
  "errors": [
    "ValueError: Staging recovery check failed."
  ],
  "failure_app": "failure-492ea24d",
  "failure_started_at": "2026-10-11T16:54:22.170331+00:00",
  "detected_at": "2026-10-11T16:54:52.429713+00:00",
  "observed_canary_status": 503,
  "failure_detected": true,
  "recovered_at": "2026-10-11T16:54:59.952629+00:00",
  "recovery_canary_status": 503,
  "recovered": false
}


RuntimeError: Check reports/finish-failure.json and the matching log in tmp. Stop this group.